# Built environment features from the site imagery

Computes per-point built environment features from the Street View images collected in
`svi_retrieval.ipynb`.

Three classes cover continuous ground and sky surfaces, so they are reported as a pixel fraction:

| Feature | Definition |
|---|---|
| `sky_ratio` | Sky pixel fraction, mean over the views of a point |
| `road_ratio` | Road pixel fraction, mean over the views |
| `sidewalk_ratio` | Sidewalk pixel fraction, mean over the views |

`gvi` stays the Green View Index from `GVIFeatureExtractor`, mean over the views.

Every other class is a discrete object, so it is reported as an instance count, summed over the
views of a point: `n_cars`, `n_facades`, `n_benches`, `n_doors`, `n_graffiti`, `n_persons`,
`n_signage`, `n_street_lights`, `n_trash`, `n_utility_poles`, `n_windows`.

A count is the number of masks the model returns for that class after `postprocess_masks` drops
masks that lie almost entirely inside a higher-confidence mask of the same class.

Inference is forced onto CPU so that this notebook reproduces the numbers computed on the
Northeastern Explorer cluster.

In [11]:
from pathlib import Path
import json

PARENT_PATH_IMAGERY = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_09_16_site_built_environment_comparison")
IMAGERY_FOLDER_NAME = "Imagery"
FEATURES_FILE_NAME = "built_environment_features.csv"

site_name_to_point_dictionaries = {}
for site_path in sorted(PARENT_PATH_IMAGERY.iterdir()):
    jsonl_path = site_path/IMAGERY_FOLDER_NAME/"point_dictionaries.jsonl"
    if not jsonl_path.exists():
        continue
    with open(jsonl_path) as f:
        site_name_to_point_dictionaries[site_path.name] = [json.loads(line) for line in f]

for site_name, point_dictionaries in site_name_to_point_dictionaries.items():
    print(f"{site_name}: {len(point_dictionaries)} points")


Carter Field: 563 points
Carter Field — 2: 587 points
DUDLEY SQUARE ROXBURY: 577 points
Gainsborough Parking Garage: 592 points
Robinson Hall: 619 points
VON HILLERN ST: 314 points
West Village B: 689 points
World Series Way : 684 points


In [ ]:
import threading

from alivestreets.features.facade_features import FacadeFeatureExtractor
from alivestreets.features.gvi_feature import GVIFeatureExtractor

FACADE_MODEL_PATH = "facades_model.pt"

_probe_extractor = FacadeFeatureExtractor()
_probe_extractor.download_model(save_path = FACADE_MODEL_PATH)

RATIO_CLASSES = ["sky", "road", "sidewalk"]
COUNT_CLASSES = [
    "vehicle", "facade", "bench", "door", "graffiti", "person",
    "signage", "street light", "trash", "utility pole", "window"
]
VEGETATION_CLASS = "vegetation"

RATIO_NAME_MAP = {"sky": "sky_ratio", "road": "road_ratio", "sidewalk": "sidewalk_ratio"}
COUNT_NAME_MAP = {
    "vehicle": "n_cars",
    "facade": "n_facades",
    "bench": "n_benches",
    "door": "n_doors",
    "graffiti": "n_graffiti",
    "person": "n_persons",
    "signage": "n_signage",
    "street light": "n_street_lights",
    "trash": "n_trash",
    "utility pole": "n_utility_poles",
    "window": "n_windows",
}

model_class_names = list(_probe_extractor.facade_feature_id_dictionary.keys())
unknown_classes = [
    name for name in RATIO_CLASSES + COUNT_CLASSES if name not in model_class_names
]
unused_classes = [
    name for name in model_class_names
    if name not in RATIO_CLASSES + COUNT_CLASSES + [VEGETATION_CLASS]
]

if unknown_classes:
    raise SystemExit(f"Classes not present in the model: {unknown_classes}")

print("Model classes:", model_class_names)
print("Ratio classes:", RATIO_CLASSES)
print("Count classes:", COUNT_CLASSES)
print("Classes left out:", unused_classes)

RATIO_FEATURES = [RATIO_NAME_MAP[name] for name in RATIO_CLASSES]
COUNT_FEATURES = [COUNT_NAME_MAP[name] for name in COUNT_CLASSES]

CONFIDENCE_THRESHOLDS = {"sky": 0.0, "road": 0.2, "facade": 0.2, "vehicle": 0.2}
DEFAULT_CONFIDENCE_THRESHOLD = 0.2

INFERENCE_DEVICE = "cpu"

_thread_local = threading.local()

def get_extractors():
    if not hasattr(_thread_local, "extractors"):
        facade_extractor = FacadeFeatureExtractor()
        facade_extractor.download_model(save_path = FACADE_MODEL_PATH, verbose = False)
        facade_extractor.model.to(INFERENCE_DEVICE)
        gvi_extractor = GVIFeatureExtractor()
        gvi_extractor.download_model(save_path = FACADE_MODEL_PATH, verbose = False)
        gvi_extractor.model.to(INFERENCE_DEVICE)
        _thread_local.extractors = (facade_extractor, gvi_extractor)
    return _thread_local.extractors

: 

In [ ]:
import os
import numpy as np
from PIL import Image

def mask_ratio(masks):
    if not masks:
        return 0.0
    final_mask = np.logical_or.reduce(masks)
    return float(np.mean(final_mask))

def compute_view_features(view_path):
    if not os.path.exists(view_path):
        return None

    try:
        with Image.open(view_path) as img:
            image = np.array(img)
    except Exception:
        return None

    facade_extractor, gvi_extractor = get_extractors()

    view_features = {}

    for class_name in COUNT_CLASSES:
        masks, _ = facade_extractor.get_masks(
            image,
            class_name = class_name,
            confidence_threshold = CONFIDENCE_THRESHOLDS.get(class_name, DEFAULT_CONFIDENCE_THRESHOLD)
        )
        view_features[COUNT_NAME_MAP[class_name]] = float(len(masks))

    for class_name in RATIO_CLASSES:
        masks, _ = facade_extractor.get_masks(
            image,
            class_name = class_name,
            confidence_threshold = CONFIDENCE_THRESHOLDS.get(class_name, DEFAULT_CONFIDENCE_THRESHOLD)
        )
        view_features[RATIO_NAME_MAP[class_name]] = mask_ratio(masks)

    view_features["gvi"] = float(gvi_extractor.compute(image)["GVI"])

    return view_features

def compute_point_features(point_dictionary):
    view_paths = point_dictionary.get("view_paths") or []
    view_features_list = [compute_view_features(path) for path in view_paths]
    view_features_list = [v for v in view_features_list if v is not None]

    n_views = len(view_features_list)
    point_features = {
        "latitude": point_dictionary.get("latitude"),
        "longitude": point_dictionary.get("longitude"),
        "n_views": n_views
    }

    if n_views == 0:
        return point_features

    for feature_name in COUNT_FEATURES:
        point_features[feature_name] = float(sum(v[feature_name] for v in view_features_list))

    for feature_name in RATIO_FEATURES + ["gvi"]:
        point_features[feature_name] = float(np.mean([v[feature_name] for v in view_features_list]))

    return point_features

: 

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
from tqdm import tqdm

MAX_WORKERS = min(4, os.cpu_count() or 1)
COORDINATE_TOLERANCE = 1e-9
CHECKPOINT_EVERY = 25
REQUIRED_FEATURES = ["n_views", "gvi"] + RATIO_FEATURES + COUNT_FEATURES

def is_complete_row(row):
    for feature in REQUIRED_FEATURES:
        value = row.get(feature)
        if value is None or pd.isna(value):
            return False
    return row["n_views"] > 0

def load_already_computed_features(output_path, point_dictionaries):
    if not output_path.exists():
        return {}

    existing_data = pd.read_csv(output_path)
    already_computed = {}
    for row in existing_data.to_dict("records"):
        point_index = int(row["point_index"])
        if point_index >= len(point_dictionaries):
            continue

        point_dictionary = point_dictionaries[point_index]
        same_latitude = abs(row["latitude"] - point_dictionary["latitude"]) < COORDINATE_TOLERANCE
        same_longitude = abs(row["longitude"] - point_dictionary["longitude"]) < COORDINATE_TOLERANCE
        if same_latitude and same_longitude and is_complete_row(row):
            row.pop("site_name", None)
            row.pop("point_index", None)
            already_computed[point_index] = row

    return already_computed

def write_features_csv(site_name, output_path, features_by_index):
    rows = []
    for point_index in sorted(features_by_index):
        row = dict(features_by_index[point_index])
        row["site_name"] = site_name
        row["point_index"] = point_index
        rows.append(row)

    features_data = pd.DataFrame(rows)
    leading_columns = ["site_name", "point_index"]
    features_data = features_data[leading_columns + [c for c in features_data.columns if c not in leading_columns]]
    features_data.to_csv(output_path, index = False)
    return features_data

site_name_to_features = {}
for site_name, point_dictionaries in site_name_to_point_dictionaries.items():
    output_path = PARENT_PATH_IMAGERY/site_name/FEATURES_FILE_NAME
    features_by_index = load_already_computed_features(output_path, point_dictionaries)
    pending_indices = [i for i in range(len(point_dictionaries)) if i not in features_by_index]

    print(f"{site_name}: {len(features_by_index)} points already computed, {len(pending_indices)} to compute")

    if pending_indices:
        with ThreadPoolExecutor(max_workers = MAX_WORKERS) as executor:
            computed_features = executor.map(
                compute_point_features, [point_dictionaries[i] for i in pending_indices]
            )
            progress = tqdm(zip(pending_indices, computed_features), total = len(pending_indices))
            for n_done, (point_index, point_features) in enumerate(progress, start = 1):
                features_by_index[point_index] = point_features
                if n_done % CHECKPOINT_EVERY == 0:
                    write_features_csv(site_name, output_path, features_by_index)

    features_data = write_features_csv(site_name, output_path, features_by_index)
    site_name_to_features[site_name] = features_data

    print(f"{site_name}: {len(features_data)} points -> {output_path}")


: 

In [ ]:
all_features_data = pd.concat(site_name_to_features.values(), ignore_index = True)
combined_path = PARENT_PATH_IMAGERY/FEATURES_FILE_NAME
all_features_data.to_csv(combined_path, index = False)

SUMMARY_FEATURES = ["gvi"] + RATIO_FEATURES + COUNT_FEATURES

site_summary = all_features_data.groupby("site_name")[SUMMARY_FEATURES].mean()
site_summary

: 

In [ ]:
import matplotlib.pyplot as plt

n_columns = 5
n_rows = int(np.ceil(len(SUMMARY_FEATURES)/n_columns))

fig, axes = plt.subplots(n_rows, n_columns, figsize = (4*n_columns, 3.2*n_rows))
axes = np.array(axes).flatten()

site_order = list(site_summary.index)
for ax, feature in zip(axes, SUMMARY_FEATURES):
    values = [all_features_data[all_features_data["site_name"] == site][feature].dropna() for site in site_order]
    ax.boxplot(values, vert = False, labels = site_order)
    ax.set_title(feature)
    ax.tick_params(labelsize = 8)

for ax in axes[len(SUMMARY_FEATURES):]:
    ax.axis("off")

plt.tight_layout()

: 

## Radar per place

One radar per site, built with `RadarChart` from `12 - Results Reconstruction`, the same helper used
for the station radars in the paper.

The built environment axes are the instance counts. Two axes are the error of the site sensor
against its nearest EPA reference station, so the figure asks whether high or variable error goes
with a particular built environment.

In [ ]:
import sys

sys.path.append("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/src/time_series")

from series_aligner import align_sensor_series_from_list
from time_series_comparison_metrics import compute_rmse, compute_correlation_coefficient

PARENT_PATH_SENSOR_COMPARISON = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_08_26_sensor_comparisons/data")
PARENT_PATH_SENSORS = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/sensors")

site_name_to_sensor_name = {
    "Carter Field": "Boston Carter Field (ours)",
    "Carter Field — 2": "Boston Carter Field (ref)",
    "Robinson Hall": "Boston Robinson Hall",
    "Gainsborough Parking Garage": "Boston Gainsborough Parking",
    "West Village B": "Boston West Village B",
    "World Series Way": "Boston World Series Way"
}

sensor_name_to_file = {
    "Boston Carter Field (ours)": "boston_carter_field/boston_carter_field.csv",
    "Boston Carter Field (ref)": "boston_carter_field_reference/MOD-00157-CarterField.csv",
    "Boston Robinson Hall": "boston_robinson_hall/boston_robinson_hall.csv",
    "Boston Gainsborough Parking": "boston_gainsborough_parking/boston_gainsborough_parking.csv",
    "Boston West Village B": "boston_west_village_b/boston_west_village_b.csv",
    "Boston World Series Way": "boston_world_series_way/boston_world_series_way.csv"
}

EPA_STATION_TO_SITE_NAME = {
    "Roxbury (EPA)": "DUDLEY SQUARE ROXBURY",
    "Von Hillern (EPA)": "VON HILLERN ST"
}

MIN_VALID_PM25 = 0.05
MAX_VALID_PM25 = 200

def convert_from_fahrenheit_to_celsius(value):
    return (value - 32)*(5/9)

temperature_data = pd.read_csv(PARENT_PATH_SENSOR_COMPARISON/"boston_temp_combined.csv")
pm25_data = pd.read_csv(PARENT_PATH_SENSOR_COMPARISON/"boston_pm25_combined.csv")

epa_station_to_temperature_data = {}
epa_station_to_pm25_data = {}
for station_name, epa_site_name in EPA_STATION_TO_SITE_NAME.items():
    station_temperature = temperature_data[temperature_data["site_name"] == epa_site_name].copy()
    station_temperature["temperature"] = station_temperature["temp_f"].map(convert_from_fahrenheit_to_celsius)
    station_temperature["timestamp"] = station_temperature["date"]
    epa_station_to_temperature_data[station_name] = station_temperature

    station_pm25 = pm25_data[pm25_data["site_name"] == epa_site_name].copy()
    station_pm25["timestamp"] = station_pm25["date"]
    epa_station_to_pm25_data[station_name] = station_pm25

sensor_name_to_data = {}
for sensor_name, relative_path in sensor_name_to_file.items():
    sensor_data = pd.read_csv(PARENT_PATH_SENSORS/relative_path)
    if "temperature" not in sensor_data.columns:
        sensor_data["temperature"] = sensor_data["temp"]

    sensor_pm25 = pd.to_numeric(sensor_data["pm25"], errors = "coerce")
    is_valid = (sensor_pm25 >= MIN_VALID_PM25) & (sensor_pm25 <= MAX_VALID_PM25)
    n_low = int((sensor_pm25.notna() & (sensor_pm25 < MIN_VALID_PM25)).sum())
    n_high = int((sensor_pm25.notna() & (sensor_pm25 > MAX_VALID_PM25)).sum())
    sensor_data["pm25"] = sensor_pm25.where(is_valid)
    print(f"{sensor_name}: dropped {n_low} samples below {MIN_VALID_PM25} and {n_high} above {MAX_VALID_PM25} ug/m3")

    sensor_name_to_data[sensor_name] = sensor_data

sensor_name_to_site_name = {}
for site_name in all_features_data["site_name"].unique():
    sensor_name = site_name_to_sensor_name.get(site_name.strip())
    if sensor_name is not None:
        sensor_name_to_site_name[sensor_name] = site_name

sensor_names = [name for name in sensor_name_to_file if name in sensor_name_to_site_name]
print()
print("sensor to imagery site:", sensor_name_to_site_name)

: 

In [ ]:
SAMPLING_INTERVAL = "24h"

def compare_sensor_with_epa_station(sensor_name, epa_station_name, variable_name, station_data):
    aligned_pair, _ = align_sensor_series_from_list(
        sensor_dataframes = [station_data, sensor_name_to_data[sensor_name]],
        variable_name = variable_name,
        time_column = "timestamp",
        sensor_names = [epa_station_name, sensor_name],
        sampling_interval = SAMPLING_INTERVAL
    )

    if len(aligned_pair) == 0:
        return {"rmse": np.nan, "coeff": np.nan, "n_days": 0}

    station_series = list(aligned_pair[epa_station_name])
    sensor_series = list(aligned_pair[sensor_name])

    return {
        "rmse": compute_rmse(station_series, sensor_series),
        "coeff": compute_correlation_coefficient(station_series, sensor_series),
        "n_days": len(aligned_pair)
    }

VARIABLE_TO_STATION_DATA = {
    "temperature": epa_station_to_temperature_data,
    "pm25": epa_station_to_pm25_data
}

metric_rows = []
for sensor_name in sensor_names:
    for epa_station_name in EPA_STATION_TO_SITE_NAME:
        for variable_name, station_store in VARIABLE_TO_STATION_DATA.items():
            metrics = compare_sensor_with_epa_station(
                sensor_name, epa_station_name, variable_name, station_store[epa_station_name]
            )
            metric_rows.append({
                "sensor_name": sensor_name,
                "site_name": sensor_name_to_site_name[sensor_name],
                "epa_station": epa_station_name,
                "variable": variable_name,
                **metrics
            })

sensor_station_metrics = pd.DataFrame(metric_rows)

for variable_name in VARIABLE_TO_STATION_DATA:
    print(f"--- {variable_name} ---")
    print(
        sensor_station_metrics[sensor_station_metrics["variable"] == variable_name]
        [["sensor_name", "epa_station", "rmse", "coeff", "n_days"]]
        .to_string(index = False)
    )
    print()

sensor_station_metrics

: 

In [ ]:
RADAR_FEATURES = COUNT_FEATURES + ["temperature_rmse", "pm25_rmse"]

RADAR_AXIS_LABELS = [
    "Cars",
    "Facades",
    "Benches",
    "Doors",
    "Graffiti",
    "People",
    "Signage",
    "Street lights",
    "Trash",
    "Utility poles",
    "Windows",
    "Temp RMSE (C)",
    "PM2.5 RMSE (ug/m3)"
]

mean_rmse_per_site = (
    sensor_station_metrics
    .groupby(["site_name", "variable"])["rmse"]
    .mean()
    .unstack("variable")
    .rename(columns = {"temperature": "temperature_rmse", "pm25": "pm25_rmse"})
)

radar_data = site_summary.join(mean_rmse_per_site, how = "inner")
radar_data = radar_data[RADAR_FEATURES].dropna(how = "all")

RADAR_MINIMUM_VALUES = [0.0]*len(RADAR_FEATURES)
RADAR_MAXIMUM_VALUES = [float(np.ceil(radar_data[feature].max()*110)/100) for feature in RADAR_FEATURES]

print("RMSE axes are the mean over the two EPA stations.")
radar_data

: 

In [ ]:
sys.path.append("/Users/legariapena.j/Documents/SV-paper-refinement/12 - Results Reconstruction/src")

import cmcrameri.cm as cmc
from RadarChart import RadarChart

RADAR_EXPORT_PATH = PARENT_PATH_IMAGERY/"site_radars.png"

radar_site_names = list(radar_data.index)
n_sites = len(radar_site_names)
n_columns = 3
n_rows = int(np.ceil(n_sites/n_columns))

fig, axes = plt.subplots(n_rows, n_columns, figsize = (7*n_columns, 7*n_rows))
axes = np.array(axes).flatten()

radar_chart = RadarChart()
for ax, site_name in zip(axes, radar_site_names):
    color = cmc.tokyo(0.15 + 0.6*(radar_site_names.index(site_name)/max(n_sites - 1, 1)))
    radar_chart.createChart(
        [list(radar_data.loc[site_name, RADAR_FEATURES])],
        RADAR_AXIS_LABELS,
        RADAR_MINIMUM_VALUES,
        RADAR_MAXIMUM_VALUES,
        [color],
        [site_name],
        fillAlpha = 0.5,
        labelFontSize = 12,
        legendFontSize = 14,
        legendLocation = "upper center",
        legendAnchor = (0.5, 0.02),
        labelOffset = 1.18,
        ax = ax
    )

for ax in axes[n_sites:]:
    ax.axis("off")

plt.tight_layout()
plt.savefig(RADAR_EXPORT_PATH, dpi = 300, bbox_inches = "tight")
print(f"Saved {RADAR_EXPORT_PATH}")


: 